In [4]:
import os

# Fixes for loading datasets on windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["DATASETS_VERBOSITY"] = "error"
os.environ["WANDB_DISABLED"] = "true" # Keeps logs clean

import numpy as np
import torch
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

print("Libraries loaded safely. GPU Available:", torch.cuda.is_available())

Libraries loaded safely. GPU Available: True


In [ ]:
print("Bypassing Hugging Face Hub and downloading raw JSONL files from GitHub...")

# URL extracted from the author's deprecated script
base_url = "https://raw.githubusercontent.com/fhamborg/NewsMTSC/6b838e00f54423c253806327a0ae24dbffa24c9e/NewsSentiment/experiments/default/datasets/newsmtsc-rw-hf/"

data_files = {
    "train": base_url + "train.jsonl",
    "validation": base_url + "dev.jsonl",
    "test": base_url + "test.jsonl"
}

# Use the native, secure JSON loader
dataset = load_dataset("json", data_files=data_files)

print("\n--- Dataset Splits ---")
print(dataset)

print("\n--- Sample Record From Training Set ---")
print(dataset['train'][0])

Bypassing Hugging Face Hub and downloading raw JSONL files from GitHub...


Generating train split: 8739 examples [00:00, 394710.73 examples/s]
Generating validation split: 343 examples [00:00, 38437.70 examples/s]
Generating test split: 803 examples [00:00, 70073.78 examples/s]


--- Dataset Splits ---
DatasetDict({
    train: Dataset({
        features: ['mention', 'polarity', 'from', 'to', 'sentence', 'id'],
        num_rows: 8739
    })
    validation: Dataset({
        features: ['mention', 'polarity', 'from', 'to', 'sentence', 'id'],
        num_rows: 343
    })
    test: Dataset({
        features: ['mention', 'polarity', 'from', 'to', 'sentence', 'id'],
        num_rows: 803
    })
})

--- Sample Record From Training Set ---
{'mention': 'Winner', 'polarity': 0, 'from': 0, 'to': 6, 'sentence': 'Winner wrote that she had a 30-minute private meeting with the Republican lawmaker’s state policy director.', 'id': 'allsides_1000_401_25_Reality Leigh Winner_0_6'}


In [ ]:
model_name = "distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_absa_function(examples):
    tokenized_inputs = tokenizer(
        examples["sentence"], 
        examples["mention"], 
        padding="max_length", 
        truncation=True, 
        max_length=128  # Safe memory limit for 8GB VRAM
    )
    # Shift labels: -1, 0, 1 -> 0, 1, 2
    tokenized_inputs["labels"] = [label + 1 for label in examples["polarity"]]
    return tokenized_inputs

print("Tokenizing entire dataset...")
tokenized_datasets = dataset.map(tokenize_absa_function, batched=True)
print("Tokenization Complete!")

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    return f1_metric.compute(predictions=predictions, references=labels, average="macro")

print("Model and metrics engine successfully initialized.")

Loading weights: 100%|██████████| 101/101 [00:00<00:00, 8753.84it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | Details
----------------------------+------------+--------
lm_head.bias                | UNEXPECTED |        
lm_head.layer_norm.weight   | UNEXPECTED |        
lm_head.dense.weight        | UNEXPECTED |        
roberta.pooler.dense.bias   | UNEXPECTED |        
lm_head.layer_norm.bias     | UNEXPECTED |        
lm_head.dense.bias          | UNEXPECTED |        
roberta.pooler.dense.weight | UNEXPECTED |        
classifier.out_proj.bias    | MISSING    |        
classifier.dense.bias       | MISSING    |        
classifier.dense.weight     | MISSING    |        
classifier.out_proj.weight  | MISSING    |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing

Model and metrics engine successfully initialized.


In [ ]:
training_args = TrainingArguments(
    output_dir="../checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)

print("Training configuration ready.")

Training configuration ready.


In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.539638,0.437217,0.833983
2,0.424340,0.406010,0.844113
3,0.236579,0.451211,0.840904


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.64it/s]


TrainOutput(global_step=1641, training_loss=0.4357221296544049, metrics={'train_runtime': 83.1561, 'train_samples_per_second': 315.274, 'train_steps_per_second': 19.734, 'total_flos': 868239931191552.0, 'train_loss': 0.4357221296544049, 'epoch': 3.0})

In [ ]:
output_model_path = "../newsmtsc_distilroberta_absa"
trainer.save_model(output_model_path)
tokenizer.save_pretrained(output_model_path)
print(f"Model successfully saved to {output_model_path}")